# Face Recognition: HybridFaceNet (CNN-Transformer Conformer-style)
**TensorFlow 2.x + ArcFace + One-shot Face Verification**

## Pipeline
1. Project Root & GPU Init
2. Model Build / Shape Sanity Check
3. Two-phase Training: CNN Warm-up ? Joint Hybrid Training
4. Verification Evaluation: CNN-only vs Trans-only vs Fused
5. Two-Image Similarity Test

**Results are saved in `models/face_recognition/hybridfacenet/results/arcface/`.**


---
## 1. Project Root & GPU Init


In [8]:
import os
import sys
import tensorflow as tf

# Change directory to project root so relative paths work properly.
while not os.path.exists('dataset_final') and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir('..')

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.append(project_root)

print(f'Working directory changed to: {os.getcwd()}')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU(s) detected: {len(gpus)}')
else:
    print('No GPU detected - running on CPU.')


Working directory changed to: c:\Users\Admin\Desktop\Project_DAT301m
GPU(s) detected: 1


---
## 2. Model Build / Shape Sanity Check

This checks the HybridFaceNet graph before long training:
- training logits: `(batch, num_classes)`
- fused/CNN/Transformer embeddings: `(batch, 512)`


In [2]:
import tensorflow as tf
from models.face_recognition.hybridfacenet.hybridfacenet import build_training_model, build_ablation_models

NUM_CLASSES = 432
training_model, backbone, arcface_layer = build_training_model(num_classes=NUM_CLASSES)

dummy_images = tf.zeros([2, 112, 112, 3], dtype=tf.float32)
dummy_labels = tf.constant([0, 1], dtype=tf.int32)
logits = training_model([dummy_images, dummy_labels], training=False)
fused_emb, cnn_emb, trans_emb = backbone(dummy_images, training=False)

print('Training logits:', logits.shape)
print('Fused embedding:', fused_emb.shape)
print('CNN embedding:', cnn_emb.shape)
print('Transformer embedding:', trans_emb.shape)
print(f'Total params: {training_model.count_params():,}')

backbone.summary()


Training logits: (2, 432)
Fused embedding: (2, 512)
CNN embedding: (2, 512)
Transformer embedding: (2, 512)
Total params: 42,106,631
Model: "HybridFaceNet_Backbone"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 image_input (InputLayer)       [(None, 112, 112, 3  0           []                               
                                )]                                                                
                                                                                                  
 preprocess (Lambda)            (None, 112, 112, 3)  0           ['image_input[0][0]']            
                                                                                                  
 stem_conv (Conv2D)             (None, 56, 56, 64)   1728        ['preprocess[0][0]']             
                                           

---
## 3. Main Training: HybridFaceNet + ArcFace

**Two-phase training:**
- Phase 1: CNN warm-up, epochs 1-5, FCU gates disabled, LR `0.01`
- Phase 2: joint training, epochs 6-30, SGD + Nesterov + weight decay, warm-up cosine LR
- Progressive ArcFace margin: `0.1 ? 0.5`, step every 5 epochs
- `--resume` restores latest full checkpoint with optimizer state and epoch

Recommended batch size:
- `64` for lower VRAM
- `96` if VRAM is >= 10GB


In [3]:
import importlib
from models.face_recognition.hybridfacenet import train
importlib.reload(train)
from models.face_recognition.hybridfacenet.train import main as train_main

sys.argv = [
    'train.py',
    '--dataset_dir', 'dataset_final/train',
    '--val_dir', 'dataset_final/val',
    '--test_dir', 'dataset_final/test',
    '--epochs', '30',
    '--warmup_epochs', '5',
    '--batch_size', '128',
    '--embedding_dim', '512',
    '--dropout', '0.3',
    '--warmup_lr', '0.01',
    '--lr', '0.05',
    '--lr_start', '1e-4',
    '--lr_min', '1e-5',
    '--weight_decay', '5e-4',
    '--arcface_scale', '64.0',
    '--patience', '15',
    '--verify_every', '5',
    '--verify_pairs', '5000',
    '--branch_val_batches', '20',
    '--resume',
]

train_main()


[GPU] Memory Growth enabled. 1 GPU(s) detected.
[Data] train=248400 val=31327 classes=432
[Model] params=42,106,631
Model: "HybridFaceNet_Backbone"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 image_input (InputLayer)       [(None, 112, 112, 3  0           []                               
                                )]                                                                
                                                                                                  
 preprocess (Lambda)            (None, 112, 112, 3)  0           ['image_input[0][0]']            
                                                                                                  
 stem_conv (Conv2D)             (None, 56, 56, 64)   1728        ['preprocess[0][0]']             
                                                            

KeyboardInterrupt: 

---
## 4. Verification Evaluation: CNN-only vs Trans-only vs Fused

Runs ablation evaluation on unseen identities and saves:
- `verification_metrics.json`
- `roc_curve_ablation.png`
- `score_distribution_cnn_only.png`
- `score_distribution_trans_only.png`
- `score_distribution_fused.png`
- `tsne_fused_embeddings.png` if `scikit-learn` is installed


In [5]:
import importlib
from models.face_recognition.hybridfacenet import evaluate
importlib.reload(evaluate)
from models.face_recognition.hybridfacenet.evaluate import main as eval_main

sys.argv = [
    'evaluate.py',
    '--test_dir', 'dataset_final/test',
    '--weights', 'models/face_recognition/hybridfacenet/results/arcface/best_hybridfacenet_backbone.h5',
    '--weights_type', 'backbone',
    '--num_classes', '432',
    '--num_pairs', '50000',
    '--batch_size', '64',
    '--save_dir', 'models/face_recognition/hybridfacenet/results/eval',
]

eval_main()


[Eval] cnn_only
[Eval] trans_only
[Eval] fused


KeyboardInterrupt: 

---
## 5. Quick Results Preview


In [ ]:
from pathlib import Path
from IPython.display import Image, display
import json

result_dir = Path('models/face_recognition/hybridfacenet/results/eval')
metrics_path = result_dir / 'verification_metrics.json'

if metrics_path.exists():
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))
else:
    print(f'Metrics file not found: {metrics_path}')

for image_name in [
    'roc_curve_ablation.png',
    'score_distribution_fused.png',
    'tsne_fused_embeddings.png',
]:
    image_path = result_dir / image_name
    if image_path.exists():
        print(image_path)
        display(Image(filename=str(image_path)))


---
## 6. Two-Image Similarity Test

Change `IMG_ORIGIN`, `IMG_SAMPLE`, and `THRESHOLD` after evaluation. Use the fused embedding by default.


In [10]:
import numpy as np
from PIL import Image
from models.face_recognition.hybridfacenet.hybridfacenet import build_inference_model

WEIGHTS = 'models/face_recognition/hybridfacenet/results/arcface/best_hybridfacenet_backbone.h5'
IMG_SIZE = 112
THRESHOLD = 0.35  # Replace with optimal_threshold from verification_metrics.json

# Build inference model and load backbone weights by name through the wrapped backbone layer.
inference_model = build_inference_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), embedding_dim=512, dropout_rate=0.3)
inference_model.get_layer('HybridFaceNet_Backbone').load_weights(WEIGHTS)
print(f'Loaded weights from: {WEIGHTS}')

def load_raw_image(path):
    img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    return np.asarray(img, dtype=np.float32)

IMG_ORIGIN = r'C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\origin.jpg'  # Replace if needed
IMG_SAMPLE = r'C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\test5.png'    # Replace if needed

img1 = np.expand_dims(load_raw_image(IMG_ORIGIN), axis=0)
img2 = np.expand_dims(load_raw_image(IMG_SAMPLE), axis=0)

emb1 = inference_model.predict(img1, verbose=0)[0]
emb2 = inference_model.predict(img2, verbose=0)[0]

similarity = float(np.dot(emb1, emb2))
is_same = similarity > THRESHOLD

print(f'Cosine similarity: {similarity:.4f}')
print(f'Threshold: {THRESHOLD:.4f}')
print('Prediction:', 'SAME identity' if is_same else 'DIFFERENT identity')


Loaded weights from: models/face_recognition/hybridfacenet/results/arcface/best_hybridfacenet_backbone.h5
Cosine similarity: 0.1690
Threshold: 0.3500
Prediction: DIFFERENT identity
